# DeBERTa-v3-large Fake News Training

Simple notebook for binary fake news detection on `GonzaloA/fake_news` using the dataset's existing `train`, `validation`, and `test` splits.


In [ ]:
# Imports
import html
import math
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    get_linear_schedule_with_warmup,
    set_seed,
)

SEED = 42
set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 1. Load and Prepare Dataset


In [ ]:
# Load dataset
ds = load_dataset('GonzaloA/fake_news')
print(ds)

URL_PATTERN = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
WHITESPACE_PATTERN = re.compile(r'\s+')

LABELS = ['FAKE', 'REAL']
id2label = {0: 'FAKE', 1: 'REAL'}
label2id = {'FAKE': 0, 'REAL': 1}

def clean_text(text):
    if text is None:
        return ''
    text = html.unescape(str(text))
    text = URL_PATTERN.sub(' ', text)
    text = HTML_TAG_PATTERN.sub(' ', text)
    text = WHITESPACE_PATTERN.sub(' ', text)
    return text.strip()

def normalize_label(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)) and not pd.isna(value):
        value = int(value)
        if value in (0, 1):
            return value
    value = str(value).strip().lower()
    if value in {'0', 'fake', 'false'}:
        return 0
    if value in {'1', 'real', 'true'}:
        return 1
    return None

def build_text(example):
    title = clean_text(example.get('title', ''))
    content = clean_text(example.get('content', example.get('text', '')))
    if title and content:
        example['text'] = f'{title} [SEP] {content}'
    elif title:
        example['text'] = title
    else:
        example['text'] = content
    example['labels'] = normalize_label(example.get('value'))
    return example

ds = ds.map(build_text)

def filter_valid(example):
    return example['labels'] is not None and len(example['text']) > 0

ds = ds.filter(filter_valid)
ds = ds.remove_columns([c for c in ds['train'].column_names if c not in ('text', 'labels')])

for split in ds:
    counts = pd.Series(ds[split]['labels']).value_counts().sort_index().to_dict()
    print(split, len(ds[split]), counts)


## 2. Tokenization


In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-large'
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tokenized = ds.map(tok, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized)


## 3. Model


In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
model.gradient_checkpointing_enable()
model.config.use_cache = False

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
    }


## 4. Training Configuration


In [ ]:
OUTPUT_DIR = 'fake_news_deberta_v3_large'

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)

optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)

train_len = len(tokenized['train'])
steps_per_epoch = math.ceil(train_len / (args.per_device_train_batch_size * args.gradient_accumulation_steps))
num_training_steps = steps_per_epoch * int(args.num_train_epochs)
num_warmup_steps = int(args.warmup_ratio * num_training_steps)

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
print(f'Training steps: {num_training_steps}, Warmup: {num_warmup_steps}')


## 5. Train Model


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    optimizers=(optimizer, scheduler),
)

train_out = trainer.train()
print(f"Training complete! Loss: {train_out.metrics['train_loss']:.4f}")


## 6. Evaluate


In [ ]:
# Evaluate
val_metrics = trainer.evaluate(tokenized['validation'])
test_metrics = trainer.evaluate(tokenized['test'])

print(f"\nValidation F1: {val_metrics['eval_f1']:.4f}")
print(f"Test F1: {test_metrics['eval_f1']:.4f}")
print(f"Test Accuracy: {test_metrics['eval_accuracy']:.4f}")

pred = trainer.predict(tokenized['test'])
preds = np.argmax(pred.predictions, axis=-1)
print('\n' + classification_report(pred.label_ids, preds, target_names=LABELS, digits=4))

cm = confusion_matrix(pred.label_ids, preds)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(np.arange(2), LABELS)
plt.yticks(np.arange(2), LABELS)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

threshold = cm.max() / 2 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'), ha='center', va='center', color='white' if cm[i, j] > threshold else 'black')

plt.tight_layout()
plt.show()


## 7. Training Curve


In [ ]:
train_loss_history = [entry['loss'] for entry in trainer.state.log_history if 'loss' in entry and 'eval_loss' not in entry]
train_loss_steps = [entry['step'] for entry in trainer.state.log_history if 'loss' in entry and 'eval_loss' not in entry]
eval_f1_history = [entry['eval_f1'] for entry in trainer.state.log_history if 'eval_f1' in entry]
eval_f1_steps = [entry['step'] for entry in trainer.state.log_history if 'eval_f1' in entry]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_loss_steps, train_loss_history, marker='o')
plt.title('Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(eval_f1_steps, eval_f1_history, marker='o')
plt.title('Validation F1')
plt.xlabel('Step')
plt.ylabel('F1')

plt.tight_layout()
plt.show()


## 8. Save Model


In [ ]:
FINAL_DIR = 'fake_news_deberta_v3_large'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Model saved to {FINAL_DIR}')
